# Описание данных

Данные - специально подготовленная выборка из полного набора, описанного здесь: https://mengtingwan.github.io/data/goodreads#datasets с дополнениями из https://www.kaggle.com/datasets/austinreese/goodreads-books:
- books: перечень книг (~44 тыс.) из списка https://www.goodreads.com/list/show/1.Best_Books_Ever (совпадение по book id),
- interactions: взаимодействие пользователей с книгами

Основной массив данных был собран в конце 2017.

# Инициализация

In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install fastparquet

Note: you may need to restart the kernel to use updated packages.


In [4]:
import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [5]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

Создаём логгер, который удобен для использования в ноутбуке.

# Load Data

Загружаем данные как они есть, начинаем на них смотреть: какие в них есть сущности, атрибуты, распределения.

In [6]:
import sys
import pandas as pd
import pyarrow as pa

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
Pandas: 3.0.6
PyArrow: 26.0.0


In [11]:
books = pd.read_parquet(
    "/workspaces/mle-recsys-start/books.parquet",
    engine="fastparquet"
)

In [8]:
books.info()

<class 'pandas.DataFrame'>
Index: 43312 entries, 3.0 to 2360448.0
Columns: 833 entries, book_id to genre_and_votes_dict.th Century
dtypes: Int64(3), bool(1), float64(816), int64(2), object(11)
memory usage: 275.4+ MB


Пример данных по книгам

In [9]:
print(books[["book_id", "author", "title", "num_pages", "average_rating"]].sample(5, random_state=52).sort_values("book_id").set_index("book_id").to_string())

                                             author                                            title  num_pages  average_rating
book_id                                                                                                                        
69882                 Heinrich Böll, Patrick Bowles                      Billiards at Half-Past Nine        288            3.93
267533                               Robert Cormier                                         8 Plus 1        173            3.67
820966    Martin Luther, J.I. Packer, O.R. Johnston                          The Bondage of the Will        322            4.17
4532622                               Karen Hancock                                      The Enclave        492            3.82
20444381                             Diana Gabaldon  Os tambores de outono - 1ª Parte (Outlander #4)       1048            3.99


In [10]:
interactions = pd.read_parquet("./goodsread/interactions.parquet")

FileNotFoundError: [Errno 2] No such file or directory: './goodsread/interactions.parquet'

Можно оценить объём данных о взаимодействиях

In [ ]:
interactions.info()

Пример данных по взаимодействиям

In [ ]:
print(interactions[["user_id", "book_id", "started_at", "read_at", "is_read", "rating"]].sample(5, random_state=52).sort_values("user_id").set_index(["user_id", "book_id"]).to_string())

# EDA

Посмотрим на данные о книгах, какие поля есть.

In [ ]:
books.sample(3).T

Посмотрим на данные о взаимодействиях пользователей с книгами.

In [ ]:
books["book_id"].nunique()

In [ ]:
# распределение книг по годам публикации

fig, axs = plt.subplots(2, 1, figsize=(12, 6))

ax = axs[0]
books["publication_year"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.tick_params(axis='x', labelsize=8)

ax = axs[1]
books.query("publication_year >= 1900 and publication_year <= 2025")["publication_year"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.tick_params(axis='x', labelsize=8)

fig.tight_layout()

# События по месяцам

In [ ]:
from pandas.tseries.offsets import MonthEnd

interactions["started_at_month"] = pd.to_datetime(interactions["started_at"]).dt.to_period('M').dt.to_timestamp().dt.date

In [ ]:
interactions_by_month = interactions.groupby("started_at_month").agg(events=("started_at", "count"), users=("user_id", "nunique")).reset_index()

In [ ]:
interactions_by_month

In [ ]:
# распределение взаимодействий по годам

fig, axs = plt.subplots(2, 1, figsize=(12, 6))

ax = axs[0]
ax.plot(interactions_by_month["started_at_month"], interactions_by_month["events"])
# ax.tick_params(axis='x', labelsize=8)
ax.set_title("#events by months")

ax = axs[1]
ax.plot(interactions_by_month["started_at_month"], interactions_by_month["users"])
# ax.tick_params(axis='x', labelsize=8)
ax.set_title("#users by months")

fig.tight_layout()

# Преобработка

Примем новые именования:
- books -> items (более обобщённое название)
- interactions -> events (несколько короче, удобнее в коде)

In [ ]:
items = books.copy()
events = interactions.copy()

In [ ]:
del books
del interactions

In [ ]:
# оставим события только до ноября 2017

event_cutoff_date = pd.to_datetime("2017-11-01").date()
events = events.query("started_at < @event_cutoff_date and read_at < @event_cutoff_date").copy()

# Ваш код здесь

# Сохранение результатов

In [ ]:
items.to_parquet("items.par")

In [ ]:
events.to_parquet("events.par")